# Session 9 — Baseline model + formalize preprocessing with sklearn Pipeline

Per `docs/Credit_Risk_Pipeline_Plan_v3.md` (buổi 9).

**Goal**: train Logistic Regression baselines for two feature sets — `portfolio` (all
features) and `at_application` (leakage columns from buổi 8 excluded) — and compare
ROC-AUC / PR-AUC.

**Before continuing past the data-loading cells below**: `model/preprocessing.py`'s
`build_preprocessor()` and `build_pipeline()` are left as `NotImplementedError` stubs —
write them yourself (see that file's docstring), don't have Claude Code fill them in.
The point of this session is being able to explain *why* `StandardScaler` fits only on
the train split and *why* `OneHotEncoder` over `LabelEncoder`, not just running code
that works.

In [ ]:
import sys

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

sys.path.insert(0, '../etl')
sys.path.insert(0, '..')
from config import get_engine
from model.features import (
    TARGET_COL,
    get_feature_columns,
    split_numeric_categorical,
)
from model.preprocessing import build_pipeline

engine = get_engine()

## Load data

Reads from the `ml_features` view (`sql/views.sql`) — already filtered to `data_source = 'historical'` and structurally excludes the dashboard-only window columns.

In [ ]:
df = pd.read_sql("SELECT * FROM ml_features", engine)
print(df.shape)
df.head()

## Feature sets

Recap of the buổi 8 decision (see `notebooks/01_eda.ipynb`, `CLAUDE.md`,
`docs/MEMORY.md`): `loan_grade` and `loan_int_rate` are near-deterministic with the
target and are outputs of underwriting, not inputs available at application time — they
stay in `LEAKAGE_COLS` (`model/features.py`) and are dropped from the
`at_application` feature set only.

In [ ]:
feature_cols = {
    "portfolio": get_feature_columns(df.columns, "portfolio"),
    "at_application": get_feature_columns(df.columns, "at_application"),
}
for name, cols in feature_cols.items():
    print(f"{name} ({len(cols)} cols): {cols}")

## Train/test split

One stratified split on `loan_status`, reused for both feature sets so the comparison
isn't confounded by different train/test rows.

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df[TARGET_COL]
)
print(f"train: {train_df.shape}, test: {test_df.shape}")

## Your answers

Write your answers to the two questions in `model/preprocessing.py`'s docstring here
before implementing the functions:

1. Why must `StandardScaler` fit only on `X_train`, after `train_test_split`?

   _(your answer)_

2. Why `OneHotEncoder` instead of `LabelEncoder` for the nominal categorical columns?

   _(your answer)_

Then implement `build_preprocessor()` and `build_pipeline()` in that file and re-run
the cells below.

## Train + evaluate both feature sets

In [ ]:
results = {}

for name, cols in feature_cols.items():
    numeric_cols, categorical_cols = split_numeric_categorical(cols)
    pipeline = build_pipeline(numeric_cols, categorical_cols)
    pipeline.fit(train_df[cols], train_df[TARGET_COL])

    proba = pipeline.predict_proba(test_df[cols])[:, 1]
    results[name] = {
        "roc_auc": roc_auc_score(test_df[TARGET_COL], proba),
        "pr_auc": average_precision_score(test_df[TARGET_COL], proba),
    }

pd.DataFrame(results).T

## Comparison

Fill in after running the cell above: how large is the gap between `portfolio` and
`at_application`? Does it match the leakage severity you found in buổi 8?